In [ ]:
%pip install websockets



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import websockets

# Move codes and the rover's address come from drive.py, beside this notebook
# in client/, where tools/check_protocol.py keeps the codes in step with the
# firmware. A hand-copied 16 here was sent as "clockwise"; 16 is
# PIVOT_SIDEWAYS_BACKWARD_LEFT.
from drive import DEFAULT_HOST, DEFAULT_PORT, MOVE_FORWARD, ROTATE_CLOCKWISE

async def websocket_communication():
    uri = f"ws://{DEFAULT_HOST}:{DEFAULT_PORT}"  # Replace with your rover's address

    # Every command switches the rover to manual mode. To let it explore again
    # afterwards, send RESUME_AUTONOMOUS (or press t in drive.py).
    async with websockets.connect(uri) as websocket:
        try:
            while True:
              response = await websocket.recv()
              response = json.loads(response)
              print(f"{response}")
              # Distances appear only once the scanner has measured every
              # bearing, so the first frames after a reset carry none.
              if "distanceFront" not in response:
                continue
              request = {
                  "speed": 75,
                  "duration": 300
              }
              # 999 means no echo: nothing within range, so the way is clear.
              if (response["distanceFront"] < 20):
                request["move"] = ROTATE_CLOCKWISE
                request["speed"] = 128
              else:
                request["move"] = MOVE_FORWARD

              request = json.dumps(request)
              print(f"{request}")
              await websocket.send(request)
              # No sleep: telemetry arrives every 500 ms and paces this loop.
              # Sleeping longer lets frames queue faster than they are read, so
              # every decision is made on older data, and once the queue fills
              # the library stops reading -- the rover's heartbeat pings go
              # unanswered and it drops the connection.
        except websockets.ConnectionClosed as e:
            print(f"Connection closed: {e}")

# Running the WebSocket client in a loop in a Jupyter notebook
await websocket_communication()
